In [2]:
import gc
import math
import os
import sys

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
%load_ext autoreload
%autoreload 2
from drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.sampler import RandomSampler

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
drugAct, pos_num, null_mask, S_d, S_c, S_g, A_cg, A_dg, _, _, _ = load_data(
    "nci", is_zero_pad=True
)

load nci
unique drugs: 177
unique genes: 251
DTI unique genes:  251
Top 90% variable genes:  2383
Total:  2582
Final gene exp shape: (60, 2582)
Final drug Act shape: (1005, 60)


100%|█████████████████████████████████████████████████████████████████████████████████████████| 25/25 [00:03<00:00,  7.77it/s]


Done!


In [7]:
PATH = "../nci_data/"

In [8]:
method = "Transformer"
study = optuna.create_study(
    storage=f"sqlite:///{method}_small.sqlite3",
    study_name=method,
    load_if_exists=True,
)

[I 2025-04-17 16:06:43,493] Using an existing study with name 'Transformer' instead of creating a new one.


In [9]:
def auto_convert_params(params, nan_replace=None):
    """Convert parameter types automatically

    Args:
        params (dict): Parameter dictionary before conversion
        nan_replace: Replacement value for NaN (default None)

    Returns:
        dict: Parameter dictionary after type conversion
    """
    converted = {}
    for k, v in params.items():
        if isinstance(v, float) and math.isnan(v):
            converted[k] = nan_replace
        elif isinstance(v, float) and v.is_integer():
            converted[k] = int(v)
        else:
            converted[k] = v
    return converted

In [10]:
params = study.best_trials[0].params
params = auto_convert_params(params, nan_replace=0)
params.update(
    {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "epochs": 100,
        "gnn_layer": method,
    }
)
params

{'dropout1': 0.15000000000000002,
 'dropout2': 0.25,
 'hidden1': 325,
 'hidden2': 304,
 'hidden3': 195,
 'heads': 2,
 'activation': 'gelu',
 'optimizer': 'AdamW',
 'lr': 0.0005154151449999027,
 'weight_decay': 0.0010982551922640643,
 'scheduler': None,
 'n_drug': 1005,
 'n_cell': 60,
 'n_gene': 2582,
 'epochs': 100,
 'gnn_layer': 'Transformer'}

In [11]:
k = 5
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()

for seed, (train_index, test_index) in enumerate(kfold.split(np.arange(pos_num))):
    sampler = RandomSampler(
        drugAct.T,
        train_index,
        test_index,
        null_mask.T,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
        PATH,
        seed,
    )
    (
        _,
        train_attention,
        val_attention,
        _,
        _,
        _,
        _,
        _,
        _,
    ) = drGAT.train(sampler, params=params, device=device, verbose=False)
    break

Using device: cuda
Best model found at epoch 100


In [12]:
val_attention

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [13]:
train_attention

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [14]:
idxs = np.load("../nci_data/idxs.npy", allow_pickle=True)
idxs

array([[0, 1, 2, ..., 3644, 3645, 3646],
       [740, 752, 755, ..., 'ZP3', 'ZSCAN18', 'ZYX']], dtype=object)

In [15]:
pd.DataFrame(train_attention, index=idxs[1], columns=idxs[1]).to_csv(
    "train_attention.csv.gz", compression="gzip"
)

In [16]:
pd.DataFrame(val_attention, index=idxs[1], columns=idxs[1]).to_csv(
    "val_attention.csv.gz", compression="gzip"
)